# 05-08 Custom Generator: 스트리밍을 유지하며 데이터 변환하기

Ch05-03에서 `RunnableLambda`로 함수를 체인에 연결하는 법을 배웠어요. 하지만 `RunnableLambda`는 입력을 한 번에 받아 한 번에 결과를 반환하기 때문에 **스트리밍(Streaming)**이 끊겨요.

제너레이터(Generator) 함수를 LCEL에 연결하면 **스트리밍을 유지하면서** 중간 데이터를 변환할 수 있어요. LLM이 토큰을 하나씩 생성하는 동안, 그 토큰을 실시간으로 가공해서 사용자에게 보여줄 수 있다는 뜻이에요.

## 학습 목표

- 파이썬 제너레이터(Generator) 함수를 LCEL 파이프라인에 연결하여 스트리밍 방식으로 데이터를 변환할 수 있어요
- 동기(Sync) 제너레이터와 비동기(Async) 제너레이터의 시그니처(Signature) 차이를 설명할 수 있어요
- 스트리밍 도중 쉼표로 구분된 문자열을 실시간으로 리스트 항목으로 변환하는 출력 파서를 만들 수 있어요

## 사전 지식

- 파이썬 제너레이터 함수와 `yield` 키워드 (아래에서 간단히 복습해요)
- `Iterator`와 `AsyncIterator` 타입 힌트
- LCEL `stream()` 메서드의 동작 방식 (Ch05-03 RunnableLambda 참고)

---

> **제너레이터 스트리밍 흐름** — 아래 다이어그램은 LLM 출력이 제너레이터를 거쳐 사용자에게 전달되는 과정을 보여줘요.

```mermaid
flowchart LR
    LLM["LLM<br/>토큰 생성"]:::process
    SC["stream()<br/>청크 단위 전달"]:::process
    GEN["제너레이터<br/>데이터 변환<br/>(yield)"]:::process
    USER["사용자<br/>실시간 수신"]:::output

    LLM -->|"'마이크로'"| SC
    SC -->|"'마이크로'"| GEN
    GEN -->|"['마이크로소프트']"| USER

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
```

**제너레이터 시그니처(Signature):**

| 종류 | 시그니처 | 실행 메서드 |
|------|---------|------------|
| 동기 제너레이터 | `Iterator[Input] -> Iterator[Output]` | `stream()` |
| 비동기 제너레이터 | `AsyncIterator[Input] -> AsyncIterator[Output]` | `astream()` |

**주요 활용:**
- 사용자 정의 출력 파서(Output Parser) 구현
- 스트리밍 기능을 유지하면서 이전 단계의 출력 수정
- 실시간 데이터 처리 및 변환

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

### 파이썬 제너레이터 빠른 복습

제너레이터가 익숙하지 않은 분을 위해 핵심만 정리할게요.

> 🔑 **핵심 개념**: 제너레이터(Generator)는 값을 하나씩 순차적으로 생산하는 함수예요. `yield` 키워드를 사용해요.

일반 함수는 `return`으로 결과를 **한 번에** 반환해요. 제너레이터는 `yield`로 값을 **하나씩** 내보내고, 다음 값이 요청될 때까지 실행을 멈춰요.

| | 일반 함수 | 제너레이터 함수 |
|---|---|---|
| 키워드 | `return` | `yield` |
| 반환 방식 | 전체 결과를 한 번에 | 값을 하나씩 순차적으로 |
| 메모리 | 결과 전체를 메모리에 저장 | 현재 값만 메모리에 유지 |
| LLM 활용 | `invoke()` — 응답 완성 후 반환 | `stream()` — 토큰 단위로 실시간 전달 |

> 🎯 **강의 포인트**: 일반 함수는 `return`으로 한 번에 결과를 반환하지만, 제너레이터는 `yield`로 값을 하나씩 내보내요. 이 차이가 스트리밍을 가능하게 하는 핵심이에요.

In [2]:
from typing import Iterator, List

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_template(
    "{company}와 유사한 5개의 회사를 쉼표로 구분된 목록으로 작성하세요."
)

# 모델 초기화
model = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")

# 문자열 체인 생성
str_chain = prompt | model | StrOutputParser()

print("=" * 60)
print("✅ 기본 체인 생성 완료")
print("=" * 60)


✅ 기본 체인 생성 완료


## 1. 기본 스트리밍

먼저 기본 스트리밍이 어떻게 동작하는지 확인해볼게요.

`stream()` 메서드는 LLM의 출력을 청크(Chunk) 단위로 실시간으로 반환해요. 사용자는 전체 응답이 완성되기 전에도 출력이 생성되는 것을 볼 수 있어요.

> 💡 **실무 팁**: 스트리밍은 사용자 경험을 개선해요. 전체 응답을 기다리지 않고 점진적으로 결과를 보여줄 수 있어요. Streamlit, FastAPI 등 실제 서비스에서 스트리밍을 적용하면 체감 응답 속도가 크게 향상돼요.

> 🎯 **강의 포인트**: `invoke()` vs `stream()`의 차이를 직접 비교해보면 스트리밍의 가치를 즉각 이해할 수 있어요. 아래 두 셀을 순서대로 실행하면서 출력 방식의 차이를 관찰해보세요.

In [3]:
# 스트리밍으로 실행
print("=" * 60)
print("📥 스트리밍 출력")
print("=" * 60)
for chunk in str_chain.stream({"company": "구글"}):
    print(chunk, end="", flush=True)
print("\n")


📥 스트리밍 출력
구글과 유사한 5개의 회사는 다음과 같습니다: 마이크로소프트, 애플, 아마존, 페이스북(메타), 바이두.



In [4]:
# invoke()로 실행 결과 확인
result = str_chain.invoke({"company": "구글"})
print("=" * 60)
print("📥 invoke() 결과")
print("=" * 60)
print(result)


📥 invoke() 결과
구글, 마이크로소프트, 애플, 아마존, 페이스북


## 2. 사용자 정의 제너레이터 구현

이제 핵심이에요. 쉼표로 구분된 문자열을 리스트로 변환하는 **커스텀 제너레이터**를 만들어볼게요.

### 버퍼(Buffer) 패턴이란?

LLM 스트리밍은 토큰 단위로 청크가 도착해요. `"마이크로"`, `"소프트"`, `","`, `" 애플"` 처럼 쉼표가 한 청크 안에 없을 수도 있어요. 그래서 **버퍼**에 청크를 쌓아두다가 쉼표를 발견하면 그때 `yield`로 내보내는 패턴을 사용해요.

```
청크 도착: "마이크로" → 버퍼: "마이크로"         (쉼표 없음, 대기)
청크 도착: "소프트,"  → 버퍼: "마이크로소프트,"   (쉼표 발견! yield)
                       → yield ["마이크로소프트"]
청크 도착: " 애플"    → 버퍼: " 애플"             (쉼표 없음, 대기)
스트림 종료           → yield ["애플"]            (남은 버퍼 반환)
```

> 🔑 **핵심 개념**: 제너레이터 함수는 `Iterator[Input] -> Iterator[Output]` 시그니처를 따르면 LCEL 파이프라인에 그대로 연결할 수 있어요. LangChain이 자동으로 스트리밍 체인으로 처리해요.

> ⚠️ **주의**: 제너레이터를 LCEL 파이프라인에 연결할 때 `RunnableLambda`로 감싸지 않아도 돼요. `|` 연산자로 직접 연결하면 LangChain이 자동으로 처리해요.

In [5]:
# ============================================================
# TODO: 쉼표로 구분된 문자열 스트림을 실시간으로 리스트 항목으로 분할하는 제너레이터를 작성하세요
# 힌트: Iterator[str] → Iterator[List[str]], 버퍼에 쉼표가 나타나면 즉시 yield
# 예상 결과: 스트리밍 중에 ['마이크로소프트'], ['애플'], ... 순서로 출력
# ============================================================

# ---------------------------------------------------
# 사용자 정의 제너레이터: 스트리밍을 유지하며 출력 형식 변환
# ---------------------------------------------------

# 제너레이터 시그니처: Iterator[Input] -> Iterator[Output]
# - 입력: LLM 출력 스트림 (문자열 청크)
# - 출력: 각 항목을 포함한 리스트
def split_into_list(input: Iterator[str]) -> Iterator[List[str]]:
    """
    쉼표로 구분된 문자열 스트림을 리스트 항목 스트림으로 변환
    
    Args:
        input: 문자열 이터레이터 (LLM 출력 스트림)
    
    Yields:
        각 항목을 포함한 리스트
    """
    buffer = ""  # 쉼표를 만날 때까지 문자를 저장할 버퍼
    
    for chunk in input:
        buffer += chunk  # 현재 청크를 버퍼에 추가
        
        # 버퍼에 쉼표가 있는 동안 반복 — 쉼표를 기준으로 항목 분리
        while "," in buffer:
            comma_index = buffer.index(",")
            # 쉼표 이전의 내용을 하나의 항목으로 즉시 반환 (스트리밍 유지)
            yield [buffer[:comma_index].strip()]
            # 쉼표 이후의 내용은 다음 반복을 위해 버퍼에 유지
            buffer = buffer[comma_index + 1:]
    
    # 마지막 남은 버퍼 내용 반환 (스트림 끝에 쉼표가 없는 마지막 항목)
    if buffer.strip():
        yield [buffer.strip()]


print("=" * 60)
print("✅ 제너레이터 함수 정의 완료")
print("=" * 60)


✅ 제너레이터 함수 정의 완료


In [6]:
# 제너레이터를 체인에 연결
list_chain = str_chain | split_into_list

print("=" * 60)
print("✅ 제너레이터 체인 생성 완료")
print("=" * 60)
print("이제 스트리밍 중에 각 항목이 개별적으로 반환됩니다.")


✅ 제너레이터 체인 생성 완료
이제 스트리밍 중에 각 항목이 개별적으로 반환됩니다.


In [7]:
# 제너레이터 체인 스트리밍 테스트
print("=" * 60)
print("📥 제너레이터 스트리밍 출력")
print("=" * 60)
for chunk in list_chain.stream({"company": "구글"}):
    print(chunk, flush=True)


📥 제너레이터 스트리밍 출력
['구글과 유사한 5개의 회사는 다음과 같습니다: 마이크로소프트']
['애플']
['아마존']
['페이스북(메타)']
['바이두.']


In [8]:
# invoke()로 실행 결과 확인
result = list_chain.invoke({"company": "구글"})

print("=" * 60)
print("📥 invoke() 결과")
print("=" * 60)
print(result)
print()
print("💡 제너레이터를 사용하면 스트리밍 기능을 유지하면서")
print("   출력 형식을 변환할 수 있습니다!")


📥 invoke() 결과
['구글', '빙', '야후', 'DuckDuckGo', '네이버']

💡 제너레이터를 사용하면 스트리밍 기능을 유지하면서
   출력 형식을 변환할 수 있습니다!


## 3. 비동기 제너레이터

동기 제너레이터를 비동기(Async) 버전으로 변환하면 `astream()`으로 비동기 스트리밍을 실행할 수 있어요.

### 왜 비동기가 필요할까요?

LLM API 호출은 네트워크 I/O 작업이에요. 동기 방식에서는 응답을 기다리는 동안 프로그램이 멈춰요. 비동기 방식에서는 대기 시간에 다른 작업(다른 사용자의 요청 처리 등)을 할 수 있어요. FastAPI 같은 비동기 서버 프레임워크에서는 비동기 제너레이터가 필수예요.

### 동기 vs 비동기 변환 규칙

| 동기 | 비동기 | 변경 사항 |
|------|--------|----------|
| `def` | `async def` | 함수 선언에 `async` 추가 |
| `for chunk in input` | `async for chunk in input` | 반복문에 `async` 추가 |
| `Iterator[T]` | `AsyncIterator[T]` | 타입 힌트 변경 |
| `stream()` | `astream()` | 실행 메서드 변경 |

> ⚠️ **주의**: 동기(sync) 제너레이터와 비동기(async) 제너레이터를 혼용하면 안 돼요. 체인의 실행 모드에 맞춰 선택하세요. `stream()`에는 동기 제너레이터, `astream()`에는 비동기 제너레이터를 사용해야 해요.

> 🎯 **강의 포인트**: 동기 제너레이터와 비동기 제너레이터의 차이는 `for` -> `async for`, `def` -> `async def`뿐이에요. 코드 구조는 동일하므로 동기 버전을 먼저 이해하면 비동기 전환이 쉬워요.

In [9]:
from typing import AsyncIterator

# ============================================================
# TODO: 동기 제너레이터를 비동기 버전으로 변환하세요
# 힌트: async def fn(input: AsyncIterator[str]) -> AsyncIterator[List[str]]: async for chunk in input: ...
# 예상 결과: astream()으로 비동기 스트리밍 출력
# ============================================================

# ---------------------------------------------------
# 비동기 제너레이터: AsyncIterator[Input] -> AsyncIterator[Output]
# ---------------------------------------------------

# 비동기 제너레이터 시그니처 차이:
# - 동기: Iterator[str] → Iterator[List[str]]  (for chunk in input)
# - 비동기: AsyncIterator[str] → AsyncIterator[List[str]]  (async for chunk in input)
async def asplit_into_list(input: AsyncIterator[str]) -> AsyncIterator[List[str]]:
    """
    비동기적으로 쉼표로 구분된 문자열을 리스트로 변환
    
    Args:
        input: 비동기 문자열 이터레이터
    
    Yields:
        각 항목을 포함한 리스트
    """
    buffer = ""
    # async for: 비동기 이터레이터에서 청크 수신 (일반 for 문 대신 사용)
    async for chunk in input:
        buffer += chunk
        while "," in buffer:
            comma_index = buffer.index(",")
            yield [buffer[:comma_index].strip()]
            buffer = buffer[comma_index + 1:]
    if buffer.strip():
        yield [buffer.strip()]


# 비동기 체인 생성 (동기 체인과 동일한 파이프 연산자 사용)
alist_chain = str_chain | asplit_into_list

print("=" * 60)
print("✅ 비동기 제너레이터 체인 생성 완료")
print("=" * 60)


✅ 비동기 제너레이터 체인 생성 완료


In [10]:
# 비동기 스트리밍 테스트
import asyncio

async def test_async_stream():
    print("=" * 60)
    print("📥 비동기 제너레이터 스트리밍 출력")
    print("=" * 60)
    async for chunk in alist_chain.astream({"company": "구글"}):
        print(chunk, flush=True)

# 실행 환경에 따라 적절히 호출
try:
    asyncio.get_running_loop()
except RuntimeError:
    asyncio.run(test_async_stream())  # 일반 파이썬 스크립트 환경
else:
    await test_async_stream()  # 이미 이벤트 루프가 실행 중인 노트북 환경


📥 비동기 제너레이터 스트리밍 출력
['구글과 유사한 5개의 회사는 다음과 같습니다: 마이크로소프트']
['애플']
['아마존']
['페이스북(메타)']
['바이두.']


In [11]:
# 비동기 invoke() 테스트
async def test_async_invoke():
    result = await alist_chain.ainvoke({"company": "구글"})
    print("=" * 60)
    print("📥 비동기 invoke() 결과")
    print("=" * 60)
    print(result)

# 실행 환경에 따라 적절히 호출
try:
    asyncio.get_running_loop()
except RuntimeError:
    asyncio.run(test_async_invoke())
else:
    await test_async_invoke()


📥 비동기 invoke() 결과
['구글', '빙', '야후', 'DuckDuckGo', '네이버']


## 정리

### 제너레이터 타입 비교

| 종류 | 시그니처 | 실행 메서드 | 사용 환경 |
|------|---------|------------|----------|
| 동기 제너레이터 | `Iterator[Input] -> Iterator[Output]` | `stream()` | 스크립트, 동기 서버 |
| 비동기 제너레이터 | `AsyncIterator[Input] -> AsyncIterator[Output]` | `astream()` | FastAPI, 비동기 서버 |

### 핵심 요점

- 제너레이터 함수는 `|` 연산자로 LCEL 파이프라인에 바로 연결할 수 있어요
- 스트리밍 기능을 유지하면서 중간 단계에서 출력 형식을 변환할 수 있어요
- 버퍼(Buffer) 패턴으로 구분자(쉼표 등)를 감지하고 항목을 분리해 실시간으로 반환해요
- 비동기 제너레이터는 `async def`와 `async for`로 작성하고 `astream()`으로 실행해요
- `RunnableLambda`는 스트리밍이 끊기지만, 제너레이터는 스트리밍을 유지하면서 변환할 수 있어요

### RunnableLambda vs Custom Generator 비교

| | `RunnableLambda` | 커스텀 제너레이터 |
|---|---|---|
| 스트리밍 지원 | 입력을 한 번에 수집 후 처리 | 청크 단위 실시간 처리 |
| 적합한 상황 | 전체 데이터가 필요한 변환 | 부분 데이터로 변환 가능한 경우 |
| 구현 방식 | 일반 함수 또는 `@chain` | `yield`를 사용하는 제너레이터 함수 |

### 다음 노트북 예고

다음 노트북(`09-Binding.ipynb`)에서는 `bind()` 메서드로 모델에 stop 토큰, OpenAI 함수(Function), 도구(Tool) 등을 런타임 전에 미리 바인딩(Binding)하는 방법을 배워요.